In [34]:
from dotenv import load_dotenv
load_dotenv()

True

In [35]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_mistralai import MistralAIEmbeddings, ChatMistralAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate


In [36]:
loader = PyPDFLoader("../data/JS_notes.pdf")
docs = loader.load()
len(docs)

36

In [37]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_docs = splitter.split_documents(docs)
len(splitted_docs)

36

In [38]:
embeddings = MistralAIEmbeddings(model="mistral-embed")

In [39]:
vectorStore = Chroma.from_documents(
    documents=splitted_docs,
    embedding=embeddings,
)

In [40]:
query = "what is closure"
data = vectorStore.similarity_search(query = query, k=3)
# len(data)

In [41]:
# data[0]
context = ""
for d in data:
    context += d.page_content + "\n"

print(context)    

26/36
🔐  Closures & Lexical Scope
Closures = when a function remembers its parent scope, even after the parent has finished.
Even after outer is done, counter still remembers count.
⚡  IIFE – Immediately Invoked Function Expression
Used to create private scope instantly.
🚀  Hoisting: Declarations vs Expressions
js
function outer() {
  let count = 0;
  return function () {
    count++;
    console.log(count);
  };
}
let counter = outer();
counter(); // 1
counter(); // 2
js
(function () {
  console.log("Runs immediately");
})();
js
hello(); // works
function hello() {
  console.log("Hi");
}
28/06/2025, 15:10 Complete JS Course Syllabus
26/36
🔐  Closures & Lexical Scope
Closures = when a function remembers its parent scope, even after the parent has finished.
Even after outer is done, counter still remembers count.
⚡  IIFE – Immediately Invoked Function Expression
Used to create private scope instantly.
🚀  Hoisting: Declarations vs Expressions
js
function outer() {
  let count = 0;
  retu

In [42]:
llm = ChatMistralAI(model="mistral-small-2506")


In [43]:
# res = llm.invoke(f"""Can you provide me the answer based on provided context of my question, context: {context} and question: {query}""")
# print(res.content)

# Chain -> context_generation | prompt | llm | strparser 


In [44]:
def get_context(query:set):
    data = vectorStore.similarity_search(query = query, k=3)
    context = ""
    for d in data:
        context += d.page_content + "\n"
    
    return {
        "context": context,
        "query": query
    }

In [49]:
prompt = PromptTemplate.from_template(
    """You are a helpful assistant and you provide answer based on the context of the question and if you don't know the answer, say "I don't know".
      Context: {context}
      Question: {query}
    """
)

In [50]:
rag_chain = get_context | prompt | llm

In [54]:
response = rag_chain.invoke("what is IIFE?") 
print(response.content)

An **IIFE (Immediately Invoked Function Expression)** is a JavaScript function that runs immediately after it is defined. It is used to create a private scope instantly, preventing variables from leaking into the global scope.

### Example of an IIFE:
```javascript
(function () {
  console.log("Runs immediately");
})();
```

### Key Points:
- It is **self-executing** (runs immediately).
- It creates a **private scope** to avoid polluting the global namespace.
- Useful for **data encapsulation** and **avoiding variable conflicts**.

Would you like a more detailed explanation or examples?
